# Day 05 · One Photo → 3D Point Cloud

Upload a street or site photo. **Depth Anything V2 (metric, outdoor)** predicts how far away every pixel is, **in metres**. Each pixel is then projected into 3D with its colour, shown in an interactive **Plotly** view, and exported as a `.ply` (to reuse on Day 17).

```
photo ──► Depth Anything V2 (metric outdoor) ──► depth map (m) ──► back-project with camera FOV ──► coloured points ──► Plotly + .ply
```

**Why the metric model?** The standard Depth Anything only knows near vs far (relative depth). The *Metric-Outdoor* version is trained on street and driving scenes and outputs **metres**, so the point cloud comes out roughly at real-world scale.

**Stack:** Python 3.12 · transformers · Gradio · Plotly · Open3D

**Photo:** change `PHOTO` below to your own Barcelona street photo. Eye level, looking down a street with buildings on both sides, works best.

## Step 1 · Load the model

`Small` (~100 MB) runs fast on the Mac GPU (`mps`). The first run downloads it from Hugging Face.

In [ ]:
import numpy as np, torch
from PIL import Image
from transformers import pipeline

MODEL = "depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf"
PHOTO = "test_peix_barcelona.jpg"      # ← swap for your own photo, e.g. "bcn.jpg"

device = "mps" if torch.backends.mps.is_available() else "cpu"
depth_pipe = pipeline("depth-estimation", model=MODEL, device=device)
print("loaded on", device)

## Step 2 · Predict depth

The model works at its own resolution, so the depth map is resized back to the photo's size with `interpolate`. Every value is a distance in metres.

In [ ]:
def predict_depth(img: Image.Image) -> np.ndarray:
    d = depth_pipe(img)["predicted_depth"]
    d = d.reshape(1, 1, *d.shape[-2:]).float()
    d = torch.nn.functional.interpolate(d, size=img.size[::-1], mode="bicubic", align_corners=False)
    return d.squeeze().cpu().numpy()          # H × W, metres

img = Image.open(PHOTO).convert("RGB")
img.thumbnail((1024, 1024))                   # keep it fast
depth = predict_depth(img)
print(f"photo {img.size}, depth {depth.min():.1f} m → {depth.max():.1f} m (median {np.median(depth):.1f} m)")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(img); ax[0].set_title("photo")
im = ax[1].imshow(depth, cmap="magma_r"); ax[1].set_title("depth (m), bright = near")
plt.colorbar(im, ax=ax[1], fraction=0.035)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Step 3 · Depth → coloured point cloud

A pinhole camera model turns each pixel `(u, v)` with depth `z` into a 3D point:

```
x = (u − W/2) · z / f        y = (v − H/2) · z / f        f = (W/2) / tan(FOV/2)
```

- **`fov_deg`** is the camera's horizontal field of view. Phones are about 65°. If walls look bent or the street looks stretched, adjust it.
- **`max_depth`** drops the sky and far junk, which the model sees as very far away.
- **`step`** samples every n-th pixel, so the 3D view stays fast (Plotly is happy up to ~100k points).
- Axes are arranged **Z-up** like Rhino: X = right, Y = into the scene, Z = up.

In [ ]:
def to_points(img, depth, step=3, max_depth=60.0, fov_deg=65.0):
    rgb = np.asarray(img.convert("RGB"))
    H, W = depth.shape
    f = (W / 2) / np.tan(np.radians(fov_deg) / 2)
    v, u = np.mgrid[0:H:step, 0:W:step]
    z = depth[v, u]
    keep = (z > 0) & (z < max_depth)
    u, v, z = u[keep], v[keep], z[keep]
    x = (u - W / 2) * z / f
    up = -(v - H / 2) * z / f
    pts = np.stack([x, z, up], axis=1)          # Z-up: (right, forward, up)
    cols = rgb[v, u] / 255.0
    return pts.astype(np.float32), cols.astype(np.float32)

pts, cols = to_points(img, depth)
print(f"{len(pts):,} points, extent {np.ptp(pts, axis=0).round(1)} m (x, y, z)")

## Step 4 · Interactive 3D view

Drag to orbit and scroll to zoom. The camera starts roughly where the photographer stood.

In [ ]:
import plotly.graph_objects as go

def plot(pts, cols, max_points=80_000):
    if len(pts) > max_points:                   # thin out for the browser
        idx = np.random.default_rng(0).choice(len(pts), max_points, replace=False)
        pts, cols = pts[idx], cols[idx]
    colors = [f"rgb({int(r*255)},{int(g*255)},{int(b*255)})" for r, g, b in cols]
    fig = go.Figure(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers",
                                 marker=dict(size=1.6, color=colors), hoverinfo="skip"))
    fig.update_layout(scene=dict(aspectmode="data",
                                 xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
                                 camera=dict(eye=dict(x=0, y=-1.2, z=0.35), up=dict(x=0, y=0, z=1))),
                      margin=dict(l=0, r=0, t=0, b=0), height=600, paper_bgcolor="white")
    return fig

plot(pts, cols)

## Step 5 · Export `.ply`

`.ply` stores points with colours. It opens in **Rhino** (`Import`), CloudCompare, MeshLab and Blender. The coordinates are in metres, Z-up.

In [ ]:
import open3d as o3d

def save_ply(pts, cols, path="cloud.ply"):
    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(pts.astype(np.float64))
    pc.colors = o3d.utility.Vector3dVector(cols.astype(np.float64))
    o3d.io.write_point_cloud(path, pc)
    return path

print("saved", save_ply(pts, cols), f"({len(pts):,} points)")

## Step 6 · Gradio app

Same pipeline wrapped in a small web app with sliders. Run the next cell to define it, then the last cell to launch. It opens inside the notebook, and also at http://127.0.0.1:7860 in your browser.

In [ ]:
import gradio as gr
from matplotlib import colormaps

def run(image, step, max_depth, fov):
    image = image.convert("RGB")
    image.thumbnail((1024, 1024))
    d = predict_depth(image)
    p, c = to_points(image, d, int(step), max_depth, fov)
    vis = colormaps["magma_r"](np.clip(d / max_depth, 0, 1))[..., :3]
    return Image.fromarray((vis * 255).astype(np.uint8)), plot(p, c), save_ply(p, c)

demo = gr.Interface(
    fn=run,
    inputs=[gr.Image(type="pil", label="Street / site photo"),
            gr.Slider(1, 8, value=3, step=1, label="Point spacing (px)"),
            gr.Slider(10, 150, value=60, label="Max depth (m)"),
            gr.Slider(40, 100, value=65, label="Camera FOV (°)")],
    outputs=[gr.Image(label="Depth (bright = near)"),
             gr.Plot(label="Point cloud"),
             gr.File(label="Download .ply")],
    title="Photo → Point Cloud",
    description="Depth Anything V2 (metric, outdoor) → coloured point cloud in metres.",
)

In [ ]:
demo.launch()

## Learnings

- **Metric vs relative depth:** metric models give real units, which matters as soon as the output goes into Rhino.
- **FOV is the hidden parameter:** without the real camera focal length, the 3D shape is only as good as the FOV guess.
- Depth from a single image is a **guess learned from millions of photos**. It's good at streets and weak at glass, sky and reflections.

## What I'd do with more time (but didn't)

- Read the real focal length from the photo's EXIF
- Mask the sky with a segmentation model instead of a depth cut-off
- Mesh it (Open3D Poisson) and import the mesh into Rhino
- Fuse several photos of the same site into one cloud